In [5]:

from pyspark.sql import functions as F

LAKEHOUSE = "Instacart_basket_LH"
SILVER = f"{LAKEHOUSE}.silver_tables"
GOLD = f"{LAKEHOUSE}.gold_tables"

# Read Silver tables
orders = spark.table(f"{SILVER}.orders_silver")
order_items = spark.table(f"{SILVER}.order_products_silver")
products = spark.table(f"{SILVER}.product_silver")
aisles = spark.table(f"{SILVER}.aisle_silver")
departments = spark.table(f"{SILVER}.department_silver")

# Reusable Gold table writer
def save_gold(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD}.{table_name}")
    )

    print(f"Saved: {GOLD}.{table_name}")


StatementMeta(, 8baaa157-ec45-459e-a2f0-c7b7fcf73d92, 7, Finished, Available, Finished, False)

In [3]:

dim_departments = departments.select(
    "department_id",
    "department"
)

save_gold(dim_departments, "dim_departments")


StatementMeta(, 36f2e480-34c5-46fe-95fe-7a01eab58029, 5, Finished, Available, Finished, False)

Saved: Instacart_basket_LH.gold_tables.dim_departments


In [4]:

# Determine which department each aisle belongs to
aisle_department_map = products.select(
    "aisle_id",
    "department_id"
).distinct()

# Validate the aisle-to-department relationship
conflicting_aisles = (
    aisle_department_map
    .groupBy("aisle_id")
    .count()
    .filter(F.col("count") > 1)
)

if conflicting_aisles.limit(1).count() > 0:
    raise ValueError(
        "Some aisles map to multiple departments. "
        "Review the product catalog before building dim_aisles."
    )

dim_aisles = (
    aisles
    .select("aisle_id", "aisle")
    .join(aisle_department_map, on="aisle_id", how="left")
    .select("aisle_id", "aisle", "department_id")
)

save_gold(dim_aisles, "dim_aisles")


StatementMeta(, 36f2e480-34c5-46fe-95fe-7a01eab58029, 6, Finished, Available, Finished, False)

Saved: Instacart_basket_LH.gold_tables.dim_aisles


In [10]:
dim_products = (products
    .join(aisles.select("aisle_id", "aisle"), "aisle_id", "left")
    .join(departments, "department_id", "left")
    .select("product_id", F.col("clean_product_name").alias("product_name"), "aisle_id", "aisle", "department_id", "department"))

save_gold(dim_products, "dim_products")


StatementMeta(, 8baaa157-ec45-459e-a2f0-c7b7fcf73d92, 12, Finished, Available, Finished, False)

Saved: Instacart_basket_LH.gold_tables.dim_products


In [7]:
fact_orders = (orders
    .filter(F.col("eval_set").isin("prior", "train"))
    .select("order_id", "user_id", "eval_set", "order_number",
            F.col("order_day_of_week").cast("int").alias("order_day_of_week"),
            F.col("order_hour_of_day").cast("int").alias("order_hour_of_day"),
            F.col("days_since_prior_order").cast("double").alias("days_since_prior_order"),
            F.col("first_order_flag")))
save_gold(fact_orders, "fact_orders")   # expect 3,346,083


StatementMeta(, 8baaa157-ec45-459e-a2f0-c7b7fcf73d92, 9, Finished, Available, Finished, False)

Saved: Instacart_basket_LH.gold_tables.fact_orders


In [9]:
fact_order_items = (
    order_items.alias("i")
    .join(fact_orders.alias("o"), "order_id", "inner")
    .select(
        "order_id", "i.product_id", "o.user_id",
        F.col("add_to_cart_order"),
        F.col("reordered"),
        "o.eval_set",
        "o.order_day_of_week", "o.order_hour_of_day", "o.days_since_prior_order"))
save_gold(fact_order_items, "fact_order_items")   



StatementMeta(, 8baaa157-ec45-459e-a2f0-c7b7fcf73d92, 11, Finished, Available, Finished, False)

Saved: Instacart_basket_LH.gold_tables.fact_order_items
